# 02 — CSMR mechanism pilot (Mini Train10K / Val3K)

Edit only the CONFIG cell, attach the 00c resource bundle and raw image/model inputs, then Run All on Kaggle. `RUN_FILTER=None` runs all six; a list selects one or any subset. This notebook reuses frozen SCRFD caches and never performs detection. The protocol addendum is embedded and SHA-checked.


In [ ]:
# CONFIG: mount 00c bundle, raw images, MiniFASNet source and official checkpoint.
RESOURCE_ROOT='/kaggle/input/.../minifasnet_binary_crossdomain_resources_v2'
CELEBA_ROOT='/kaggle/input/.../CelebA_Spoof'
LCC_ROOT='/kaggle/input/.../LCC_FASD_evaluation'
MINIFASNET_SOURCE_PATH='/kaggle/input/.../MiniFASNet.py'
MINIFASNET_CHECKPOINT_PATH='/kaggle/input/.../2.7_80x80_MiniFASNetV2.pth'
OUTPUT_ROOT='/kaggle/working/csmr_mechanism_pilot_10k_v1'
RUN_FILTER=None # None: all six; e.g. ['P1_worst_ce_lambda0'] or any valid subset.
BASELINE_C_DIR=None # Optional completed C_copy_pretrained run directory; metrics read-only.
RESUME=True; NUM_WORKERS=4; AMP=True
DATA_SEED=43; TRAIN_SEED=100
BATCH_SIZE=256; MAX_EPOCHS=20; EARLIEST_EARLY_STOP_EPOCH=8; EARLY_STOP_PATIENCE=4
CSMR_BAND_CENTERS=[.15,.45,.75]
CSMR_BAND_SIGMA=.10; CSMR_PRESERVE_DC=True; SELECTION_CHUNK_SIZE=32
CROP_SCALE=2.7; INPUT_SIZE=80; METRIC_TIE_EPS=1e-12
EXPECTED_CONTRACT_SHA='2f1f6eb6351550d491e4fb59e2151b1ccd9b000c0985cd32f12621f9d58c5f91'
EXPECTED_PARENT_SHA='aa67032a79627636d967dc885202540a34410c2b4d64522a5604d3aedff85b2d'
EXPECTED_DETECTOR_SHA='5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a'
EXPECTED_MODEL_SHA='e498c4ec5e1ddfaba62b941a126c19d65aa564999f3309661fe43ee8bf38acd7'
EXPECTED_PRETRAINED_SHA='a5eb02e1843f19b5386b953cc4c9f011c3f985d0ee2bb9819eea9a142099bec0'


In [ ]:
PROTOCOL_SHA='184c9ed75ac79dee60abae2611f669a9f15faec07cc6d9b38ac8243bbf68629f'
PROTOCOL_TEXT="# CSMR mechanism pilot protocol v1\n\nStatus: frozen design for the six-run, PAD-pretrained Mini Train10K/Val3K mechanism diagnostic. Date: 2026-10-02. This file is an **addendum** to `PAD_DATA_AND_EVALUATION_CONTRACT_v2_CROSSDOMAIN_BINARY.md` (SHA256 `2f1f6eb6351550d491e4fb59e2151b1ccd9b000c0985cd32f12621f9d58c5f91`), which continues to govern data, localization, model input, candidate accounting, metrics, and evaluation isolation. It authorizes only the spectral-training variations below.\n\n## Research question and controls\n\nWhy did training-only CSMR fail to add sufficient benefit to the binary, auxiliary, literature-adapted C baseline in the reported mini pilot? Do not select a winner or tune any setting on LCC. P1–P6 all start from official PAD-pretrained MiniFASNetV2 features and newly initialized binary/auxiliary heads with seed 100. They use the exact same ordered 00c Mini Train10K/Val3K manifests, seed 43, raw SCRFD cache, 2.7× crop, 80×80 BGR [0,1], bbox jitter, copied augmentation, SGD 0.005/momentum 0.9/weight decay 5e-4, batch 256, MultiStepLR [6,14] gamma 0.2, at most 20 epochs, earliest stop 8, patience 4, and Mini-Val3K-only minimum-ACER checkpoint/threshold selection. Auxiliary losses are clean-view only: `L_aux=0.1 L_spoof+0.1 L_lighting+L_attr`, with attribute BCEWithLogits only on Real samples and differentiable zero for an all-Attack batch.\n\nAll active runs use the same LOW/MID/HIGH smooth Gaussian radial spectral generator (centers [0.15,0.45,0.75], sigma 0.10, DC preserved, phase unchanged, common channel gain, attenuation only), and sample-specific worst-band selection by smallest binary aligned PAD margin. Candidate forwards use `eval()` and `no_grad()` and cannot update BN. Restore training mode for differentiable clean and selected-worst forwards. No FFT, frequency branch, selector, margin, or KL is used at inference.\n\n## Frozen run matrix\n\nLet `L_c=CE(z_clean,y_binary)`, `L_w=CE(z_worst,y_binary)`, `d=z_real-z_attack`, `m=d` for Real and `m=-d` for Attack. Let `L_margin=mean(SmoothL1(m_worst,m_clean.detach())·1[m_worst<m_clean.detach()])`. For P6, `L_KL=KL(softmax(z_clean).detach() || softmax(z_worst))` computed as `F.kl_div(log_softmax(z_worst),softmax(z_clean).detach(),reduction='batchmean')`.\n\n| Run ID | Warm-up | Gain | Active objective | Mechanism |\n|---|---:|---|---|---|\n| `P1_worst_ce_lambda0` | 1 | [0.65,0.90] | `0.5 L_c + 0.5 L_w + L_aux` | Worst-view classification without margin penalty |\n| `P2_csmr_lambda01` | 1 | [0.65,0.90] | `0.5 L_c + 0.5 L_w + 0.1 L_margin + L_aux` | Reduced harmful-margin strength |\n| `P3_mild_worst_ce_lambda0` | 1 | [0.65,0.90] | `0.75 L_c + 0.25 L_w + L_aux` | Reduced worst-view CE pressure; compare with P1 |\n| `P4_mild_gain_lambda01` | 1 | [0.80,0.95] | `0.5 L_c + 0.5 L_w + 0.1 L_margin + L_aux` | Milder attenuation; differs from P2 only in gain |\n| `P5_warmup3_lambda01` | 3 | [0.65,0.90] | `0.5 L_c + 0.5 L_w + 0.1 L_margin + L_aux` | Longer clean warm-up; differs from P2 only in duration |\n| `P6_generic_kl_lambda01` | 1 | [0.65,0.90] | `0.5 L_c + 0.5 L_w + 0.1 L_KL + L_aux` | Generic consistency with **no** harmful-margin penalty |\n\nEvery warm-up epoch uses `L_c+L_aux`. Warm-up epochs cannot become the final best checkpoint or increment active patience. The first active epoch initializes the eligible best state. Active epochs use minimum Val ACER, tie highest Val AUC, with an explicit no-improvement counter. All six retain the same worst-band selection; P6 changes only the regularizer relative to P2.\n\n## Evaluation and interpretation\n\nFreeze the selected Val checkpoint and minimum-ACER Val threshold before evaluating official CelebA Test once and official LCC once. Test/LCC threshold metrics use that locked source threshold; report AUC, EER, TPR@FPR1%, APCER, BPCER, ACER/HTER, accuracy, candidate_n, scored_n, and coverage. LCC is pilot evidence only. Preserve failed detections in candidate predictions; never use LCC for checkpoint, threshold, lambda, gain, or formulation selection. Interpret C vs P1, P1 vs P2, P1 vs P3, P2 vs P4, P2 vs P5, and P2 vs P6 as the planned mechanism contrasts. C may appear only as an optional read-only metrics reference after matching resource/model/data fingerprints; do not retrain C in this notebook.\n\n## Diagnostics and provenance\n\nPer active epoch retain clean/worst CE, weights, margin and KL terms and coefficients, harmful fraction, mean margins/drop, mean harmful-only drop (zero when none), spectral flip rate (clean correct to worst incorrect), gain bounds/mean, selected LOW/MID/HIGH counts/fractions, auxiliary components, LR, total loss, and Val metrics. Each run owns its own config hash, initialization audit, best/last checkpoints, optimizer/scheduler/AMP/RNG state, warm-up checkpoint/metrics, patience state, frozen evaluation record, predictions, and completion marker. Unknown run IDs or mismatched resume/resource/config fingerprints fail. Batch 256 OOM is fatal; a changed batch size requires a new explicit protocol. No actual outcome is implied by this design.\n"

import copy, csv, hashlib, importlib.util, inspect, json, math, os, random, time
from collections import Counter, OrderedDict
from pathlib import Path
import albumentations as A
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import Dataset, DataLoader

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()
def hash_json(x): return hashlib.sha256(json.dumps(x,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
def ordered_sha(keys): return hashlib.sha256('\n'.join(map(str,keys)).encode()).hexdigest()
def read_json(path): return json.loads(Path(path).read_text(),parse_constant=lambda s: (_ for _ in ()).throw(ValueError(s)))
def write_json(path,x): Path(path).write_text(json.dumps(x,indent=2,sort_keys=True,allow_nan=False)+'\n')
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
def state_sha(state):
    h=hashlib.sha256()
    for k in sorted(state):
        v=state[k].detach().cpu().contiguous();h.update(k.encode());h.update(str(v.dtype).encode());h.update(str(tuple(v.shape)).encode());h.update(v.numpy().tobytes())
    return h.hexdigest()
def save_torch(path,value):
    tmp=Path(str(path)+'.tmp');torch.save(value,tmp);os.replace(tmp,path)
def require_record(rec,key):
    if rec.get('status')!='VALID' or rec.get('usable_for_pad') is not True or rec.get('bbox_origin')!='SCRFD' or rec.get('source_key')!=key or not str(rec.get('bbox_source','')).startswith('SCRFD'):raise RuntimeError('Unscorable/non-SCRFD: '+key)
    box=rec.get('bbox_xyxy')
    if box is None or len(box)!=4 or 'crop_factor' in rec:raise RuntimeError('Not a raw bbox: '+key)
    x1,y1,x2,y2=map(float,box)
    if not np.isfinite(box).all() or x2<=x1 or y2<=y1:raise RuntimeError('Invalid bbox: '+key)
    return rec

def bbox_jitter(rec):
    x1,y1,x2,y2=map(float,rec['bbox_xyxy']);w,h=x2-x1,y2-y1
    if random.random()<.20:
        base=max(w,h);cx,cy=(x1+x2)/2,(y1+y2)/2;scale=random.uniform(.95,1.05)
        cx+=random.uniform(-.05,.05)*base;cy+=random.uniform(-.05,.05)*base;w*=scale;h*=scale
        return [cx-w/2,cy-h/2,cx+w/2,cy+h/2]
    return [x1,y1,x2,y2]
def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Exact 03b CropImage-compatible geometry.
    src_h,src_w=image.shape[:2];x1,y1,x2,y2=map(float,bbox_xyxy);box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:raise ValueError('Invalid crop geometry')
    used=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale));new_w,new_h=box_w*used,box_h*used
    cx,cy=box_w/2+x1,box_h/2+y1;left,top=cx-new_w/2,cy-new_h/2;right,bottom=cx+new_w/2,cy+new_h/2
    if left<0:right-=left;left=0
    if top<0:bottom-=top;top=0
    if right>src_w-1:left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1:top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:raise RuntimeError('Crop escaped image')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty crop')
    return cv2.resize(patch,(size,size))
def mild_aug(patch):
    if random.random()<.5:patch=cv2.flip(patch,1)
    if random.random()<.3:
        alpha=random.uniform(.9,1.1);beta=random.uniform(-.1,.1)*255
        patch=np.clip(patch.astype(np.float32)*alpha+beta,0,255).astype(np.uint8)
    return patch
# The literature's augmentation ranges are fixed. Support both documented Albumentations APIs.
# ISONoise works in RGB/HSV; convert only within augmentation, then restore BGR for MiniFASNet.
iso_params=inspect.signature(A.ISONoise).parameters
brightness_params=inspect.signature(A.RandomBrightnessContrast).parameters
blur_params=inspect.signature(A.MotionBlur).parameters
if {'color_shift','intensity'}<=set(iso_params) and {'brightness_limit','contrast_limit'}<=set(brightness_params) and 'blur_limit' in blur_params:
    AUGMENTATION_API='legacy_limits'
    iso=A.ISONoise(color_shift=(.15,.35),intensity=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_limit=.2,contrast_limit=.2,brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_limit=5,p=.2)
elif {'color_shift_range','intensity_range'}<=set(iso_params) and {'brightness_range','contrast_range'}<=set(brightness_params) and 'blur_range' in blur_params:
    AUGMENTATION_API='range_parameters'
    iso=A.ISONoise(color_shift_range=(.15,.35),intensity_range=(.2,.5),p=.2)
    bright=A.RandomBrightnessContrast(brightness_range=(-.2,.2),contrast_range=(-.2,.2),brightness_by_max=True,p=.3)
    motion=A.MotionBlur(blur_range=(3,5),p=.2)
else:raise RuntimeError('Unsupported Albumentations augmentation API; freeze and audit a compatible version')
COPY_AUG=A.Compose([A.HorizontalFlip(p=.5),iso,bright,motion])
def copied_aug_bgr(patch):
    rgb=cv2.cvtColor(patch,cv2.COLOR_BGR2RGB)
    augmented=COPY_AUG(image=rgb)['image']
    return cv2.cvtColor(augmented,cv2.COLOR_RGB2BGR)
def tensor_of(patch):
    if patch.shape!=(80,80,3) or patch.dtype!=np.uint8:raise ValueError('Expected BGR uint8 80x80')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)

RUNS=OrderedDict([
 ('P1_worst_ce_lambda0',{'warmup_epochs':1,'gain':[.65,.90],'clean_ce_weight':.5,'worst_ce_weight':.5,'margin_lambda':0.0,'consistency_lambda':0.0,'regularizer':'none'}),
 ('P2_csmr_lambda01',{'warmup_epochs':1,'gain':[.65,.90],'clean_ce_weight':.5,'worst_ce_weight':.5,'margin_lambda':.1,'consistency_lambda':0.0,'regularizer':'harmful_margin'}),
 ('P3_mild_worst_ce_lambda0',{'warmup_epochs':1,'gain':[.65,.90],'clean_ce_weight':.75,'worst_ce_weight':.25,'margin_lambda':0.0,'consistency_lambda':0.0,'regularizer':'none'}),
 ('P4_mild_gain_lambda01',{'warmup_epochs':1,'gain':[.80,.95],'clean_ce_weight':.5,'worst_ce_weight':.5,'margin_lambda':.1,'consistency_lambda':0.0,'regularizer':'harmful_margin'}),
 ('P5_warmup3_lambda01',{'warmup_epochs':3,'gain':[.65,.90],'clean_ce_weight':.5,'worst_ce_weight':.5,'margin_lambda':.1,'consistency_lambda':0.0,'regularizer':'harmful_margin'}),
 ('P6_generic_kl_lambda01',{'warmup_epochs':1,'gain':[.65,.90],'clean_ce_weight':.5,'worst_ce_weight':.5,'margin_lambda':0.0,'consistency_lambda':.1,'regularizer':'kl_consistency'})])
if RUN_FILTER is not None and (not isinstance(RUN_FILTER,(list,tuple)) or len(RUN_FILTER)==0 or len(set(RUN_FILTER))!=len(RUN_FILTER) or set(RUN_FILTER)-set(RUNS)):
    raise ValueError('RUN_FILTER must be None or a list/tuple of unique P1-P6 run IDs')
if NUM_WORKERS<1:raise ValueError('NUM_WORKERS must be >=1 so each worker receives a deterministic augmentation seed')
root=Path(RESOURCE_ROOT);out=Path(OUTPUT_ROOT);source=Path(MINIFASNET_SOURCE_PATH);pretrained=Path(MINIFASNET_CHECKPOINT_PATH)
for p in [root/'config.json',root/'provenance.json',root/'manifest_fingerprints.json',root/'PAD_DATA_AND_EVALUATION_CONTRACT_v2_CROSSDOMAIN_BINARY.md',
  root/'labels/celeba_auxiliary_labels.csv',root/'bbox/celeba_bbox_cache_full.json',root/'bbox/lcc_official_evaluation_bbox_cache.json',
  root/'manifests/mini_train10k.csv',root/'manifests/mini_val3k.csv',root/'manifests/crossdomain_celeba_source_dev_full.csv',
  root/'manifests/lcc_official_evaluation_full.csv',source,pretrained]:
    if not p.is_file():raise FileNotFoundError(p)
if sha(root/'PAD_DATA_AND_EVALUATION_CONTRACT_v2_CROSSDOMAIN_BINARY.md')!=EXPECTED_CONTRACT_SHA:raise RuntimeError('V2 contract mismatch')
if sha(source)!=EXPECTED_MODEL_SHA or sha(pretrained)!=EXPECTED_PRETRAINED_SHA:raise RuntimeError('Model source/pretrained hash mismatch')
config=read_json(root/'config.json');prov=read_json(root/'provenance.json');fps=read_json(root/'manifest_fingerprints.json')
if config['v2_contract_sha256']!=EXPECTED_CONTRACT_SHA or prov['v2_contract_sha256']!=EXPECTED_CONTRACT_SHA or prov['parent_contract_sha256']!=EXPECTED_PARENT_SHA:raise RuntimeError('Resource contract mismatch')
if prov['detector_sha256']!=EXPECTED_DETECTOR_SHA or prov['scrfd_execution_provider']!='CUDAExecutionProvider' or prov['mini_data_seed']!=DATA_SEED:raise RuntimeError('Resource provenance mismatch')
if config['input']!={'scale':2.7,'size':[80,80],'channels':'BGR','range':'[0,1]'}:raise RuntimeError('Input contract mismatch')
for split in ('train','test'):
    metadata=Path(CELEBA_ROOT)/'metas/intra_test'/(split+'_label.json')
    if not metadata.is_file() or sha(metadata)!=prov['source_hashes'][split+'_label_json']:raise RuntimeError('Raw CelebA metadata hash mismatch: '+split)
if (Path(LCC_ROOT)/'LCC_FASD_evaluation').is_dir():LCC_ROOT=str(Path(LCC_ROOT)/'LCC_FASD_evaluation')
for kind in ('celeba','lcc'):
    if sha(root/'bbox'/(kind+'_bbox_cache_full.json' if kind=='celeba' else 'lcc_official_evaluation_bbox_cache.json'))!=prov[kind+'_bbox_sha256']:
        # CelebA file has a distinct historical name.
        if kind!='celeba' or sha(root/'bbox/celeba_bbox_cache_full.json')!=prov['celeba_bbox_sha256']:raise RuntimeError('Cache SHA mismatch')
records=read_json(root/'bbox/celeba_bbox_cache_full.json')['records'];lcc_records=read_json(root/'bbox/lcc_official_evaluation_bbox_cache.json')['records']
for cache in (records,lcc_records):
    for key,rec in cache.items():
        if rec.get('usable_for_pad'):require_record(rec,key)
def load_frame(name,key='sample_key'):
    p=root/'manifests'/(name+'.csv');f=pd.read_csv(p,keep_default_na=False);fp=fps[name]
    if sha(p)!=fp['file_sha256'] or len(f)!=fp.get('sample_count',fp.get('candidate_n')):raise RuntimeError('Manifest SHA/count mismatch: '+name)
    if key=='sample_key' and ordered_sha(f[key])!=fp['ordered_sample_key_sha256']:raise RuntimeError('Manifest order mismatch: '+name)
    if key=='path' and ordered_sha(f[key])!=fp['ordered_path_sha256']:raise RuntimeError('LCC manifest order mismatch')
    if f[key].duplicated().any():raise RuntimeError('Duplicate manifest key: '+name)
    return f
train=load_frame('mini_train10k');val=load_frame('mini_val3k');test=load_frame('crossdomain_celeba_source_dev_full');lcc=load_frame('lcc_official_evaluation_full','path')
if len(train)!=10000 or len(val)!=3000 or len(test)!=67170 or len(lcc)!=7580 or int((lcc.label==0).sum())!=314 or int((lcc.label==1).sum())!=7266:raise RuntimeError('Population invariant')
if set(train.sample_key)&set(val.sample_key) or set(train.subject_id.astype(str))&set(val.subject_id.astype(str)):raise RuntimeError('Mini leakage')
for frame in (train,val,test):
    for row in frame.itertuples(index=False):
        rec=records[str(row.sample_key)]
        if str(row.localization_status)!=rec['status'] or bool(row.usable_for_pad)!=bool(rec['usable_for_pad']) or str(row.bbox_origin)!=rec['bbox_origin']:raise RuntimeError('Manifest/cache mismatch')
        if rec['usable_for_pad']:require_record(rec,str(row.sample_key))
for row in lcc.itertuples(index=False):
    rec=lcc_records[str(row.path)]
    if str(row.status)!=rec['status'] or bool(row.usable_for_pad)!=bool(rec['usable_for_pad']):raise RuntimeError('LCC cache mismatch')
aux=pd.read_csv(root/'labels/celeba_auxiliary_labels.csv',keep_default_na=False).set_index('sample_key',verify_integrity=True)
if sha(root/'labels/celeba_auxiliary_labels.csv')!=prov['auxiliary_labels_sha256']:raise RuntimeError('Aux SHA mismatch')
for frame in (train,val,test):
    if not set(frame.sample_key)<=set(aux.index):raise RuntimeError('Missing auxiliary labels')
for frame in (train,val):
    if not frame.usable_for_pad.astype(bool).all() or set(frame.bbox_origin)!={'SCRFD'} or set(frame.three_class_label)!={0,1,2}:raise RuntimeError('Mini data invariant')
if not torch.cuda.is_available():raise RuntimeError('Kaggle GPU required; no CPU fallback for this experiment')
print('GPU',torch.cuda.get_device_name(0),'PyTorch',torch.__version__,'CUDA',torch.version.cuda,'Albumentations',A.__version__,'OpenCV',cv2.__version__)
if hashlib.sha256(PROTOCOL_TEXT.encode()).hexdigest()!=PROTOCOL_SHA:raise RuntimeError('Embedded mechanism protocol SHA mismatch')
print('Resource/source hashes:',prov['source_hashes'],'manifest hashes:',{k:v['file_sha256'] for k,v in fps.items()},'model',sha(source),'pretrained',sha(pretrained))
out.mkdir(parents=True,exist_ok=True);(out/'runs').mkdir(exist_ok=True)
source_spec=importlib.util.spec_from_file_location('local_minifasnet_v2',source);module=importlib.util.module_from_spec(source_spec);source_spec.loader.exec_module(module)
device=torch.device('cuda');use_amp=bool(AMP)

class PADNet(nn.Module):
    def __init__(self,copied):
        super().__init__();self.copied=copied
        self.trunk=module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),drop_p=.2,num_classes=3,img_channel=3)
        if copied:
            self.trunk.prob=nn.Identity()
            self.pad_head=nn.Linear(128,2);self.spoof_type_head=nn.Linear(128,11)
            self.lighting_head=nn.Linear(128,5);self.attributes_head=nn.Linear(128,40)
    def features(self,x):
        t=self.trunk
        for layer in (t.conv1,t.conv2_dw,t.conv_23,t.conv_3,t.conv_34,t.conv_4,t.conv_45,t.conv_5,t.conv_6_sep,t.conv_6_dw,t.conv_6_flatten):x=layer(x)
        if t.embedding_size!=512:x=t.linear(x)
        x=t.drop(t.bn(x))
        if x.ndim!=2 or x.shape[1]!=128:raise RuntimeError('MiniFAS representation is not [B,128]')
        return x
    def forward(self,x,aux=False):
        z=self.features(x)
        if not self.copied:return self.trunk.prob(z)
        pad=self.pad_head(z)
        if aux:return pad,self.spoof_type_head(z),self.lighting_head(z),self.attributes_head(z)
        return pad

def init_model(spec):
    seed_all(TRAIN_SEED);model=PADNet(spec['copied']);audit={'initialization':spec['init'],'checkpoint_sha256':sha(pretrained),'source_sha256':sha(source)}
    if spec['init']=='pretrained':
        raw=torch.load(pretrained,map_location='cpu',weights_only=True)
        if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):raise RuntimeError('Official checkpoint format mismatch')
        stripped=OrderedDict((k[7:],v) for k,v in raw.items())
        if tuple(stripped['prob.weight'].shape)!=(3,128) or tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):raise RuntimeError('Official checkpoint architecture mismatch')
        target=model.trunk.state_dict();compatible={k:v for k,v in stripped.items() if k!='prob.weight'}
        bad=[k for k,v in compatible.items() if k not in target or tuple(v.shape)!=tuple(target[k].shape)]
        absent=[k for k in target if k not in compatible and k!='prob.weight']
        if bad or absent:raise RuntimeError(f'Pretrained key mismatch: {bad[:5]} {absent[:5]}')
        missing,unexpected=model.trunk.load_state_dict(compatible,strict=False)
        if unexpected or (spec['copied'] and missing) or (not spec['copied'] and set(missing)!={'prob.weight'}):raise RuntimeError('Pretrained load mismatch')
        # Reinitialize historical classifier too; copied heads were initialized by seed 100.
        if not spec['copied']:model.trunk.prob.reset_parameters()
        audit.update({'loaded_keys':sorted(compatible),'skipped_keys':['prob.weight'],'missing_keys':list(missing),'unexpected_keys':list(unexpected)})
    else:audit.update({'loaded_keys':[],'skipped_keys':[],'missing_keys':[],'unexpected_keys':[]})
    audit['initial_state_sha256']=state_sha(model.state_dict())
    with torch.no_grad():
        model.eval();assert model(torch.zeros(2,3,80,80)).shape==(2,2 if spec['copied'] else 3)
    return model,audit

class CelebaData(Dataset):
    def __init__(self,frame,training=False,copied=False):self.rows=frame.reset_index(drop=True);self.training=training;self.copied=copied
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.sample_key);rec=require_record(records[key],key)
        image=cv2.imread(str(Path(CELEBA_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        patch=mini_crop_bgr(image,bbox_jitter(rec) if self.training else rec['bbox_xyxy'])
        if self.training:patch=copied_aug_bgr(patch) if self.copied else mild_aug(patch)
        a=aux.loc[key]
        return tensor_of(patch),int(row.three_class_label),int(row.binary_label),int(a.spoof_type),int(a.lighting),torch.tensor([float(a[f'attr_{j:02d}']) for j in range(40)]),key
class LCCData(Dataset):
    def __init__(self,frame):self.rows=frame.loc[frame.usable_for_pad.astype(bool)].reset_index(drop=True)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i];key=str(row.path);rec=require_record(lcc_records[key],key)
        image=cv2.imread(str(Path(LCC_ROOT)/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(key)
        return tensor_of(mini_crop_bgr(image,rec['bbox_xyxy'])),int(row.label),key

def loader(ds,batch,shuffle=False,epoch=0):
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    def worker_init(wid):
        s=TRAIN_SEED+epoch*1000+wid;random.seed(s);np.random.seed(s);torch.manual_seed(s)
        if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(s)
    return DataLoader(ds,batch_size=batch,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,drop_last=False,generator=gen,worker_init_fn=worker_init)

def pad_score(logits):
    z=torch.as_tensor(logits)
    if z.shape[1]==3:return (z[:,0]-torch.logsumexp(z[:,1:],dim=1)).float().cpu().numpy()
    if z.shape[1]==2:return (z[:,0]-z[:,1]).float().cpu().numpy()
    raise ValueError('Expected two or three PAD logits')
def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:raise ValueError('Val needs all three classes')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0);nr=int(real.sum());na=len(y)-nr
    bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)];rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]
def metrics(y3,scores,threshold,candidate_n):
    y=np.asarray(y3,dtype=int);s=np.asarray(scores,dtype=float);real=y==0
    if len(y)!=len(s) or set(real)!={False,True} or not np.isfinite(s).all():raise ValueError('Invalid metric population')
    fpr,tpr,thr=roc_curve(real.astype(int),s);eer_idx=int(np.argmin(abs(fpr-(1-tpr))))
    pred=s>=threshold;apcer=float(pred[~real].mean());bpcer=float((~pred[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),s)),'eer':float(.5*(fpr[eer_idx]+1-tpr[eer_idx])),
      'tpr_at_fpr_1pct':float(np.interp(.01,fpr,tpr)),'apcer':apcer,'bpcer':bpcer,'acer':.5*(apcer+bpcer),
      'hter':.5*(apcer+bpcer),'binary_accuracy':float((pred==real).mean()),'locked_source_threshold':float(threshold),
      'candidate_n':int(candidate_n),'scored_n':len(y),'detector_coverage':len(y)/candidate_n}
def infer(model,dl,lcc_mode=False):
    model.eval();keys=[];ys=[];zs=[]
    with torch.no_grad():
        for batch in dl:
            if lcc_mode:x,y,k=batch
            else:x,y,_,_,_,_,k=batch
            with torch.cuda.amp.autocast(enabled=use_amp):z=model(x.to(device,non_blocking=True))
            keys.extend(k);ys.extend(y.numpy().tolist());zs.append(z.float().cpu().numpy())
    if not zs:raise RuntimeError('No scorable evaluation samples')
    return list(map(str,keys)),np.asarray(ys,dtype=int),np.concatenate(zs)

def radial_masks(size=INPUT_SIZE,device=device):
    # Same radial geometry as the tested 03b CSMR notebook.
    y=torch.arange(size,device=device,dtype=torch.float32)-size//2
    yy,xx=torch.meshgrid(y,y,indexing='ij')
    radius=torch.sqrt(xx.square()+yy.square())/math.sqrt(2*(size//2)**2)
    centers=torch.as_tensor(CSMR_BAND_CENTERS,dtype=torch.float32,device=device)
    masks=torch.exp(-((radius[None]-centers[:,None,None])**2)/(2*CSMR_BAND_SIGMA**2))
    if masks.shape!=(3,size,size) or not torch.isfinite(masks).all():raise RuntimeError('Invalid CSMR radial masks')
    return masks
def spectral_views(x,gains,masks):
    if x.ndim!=4 or x.shape[1:]!=(3,80,80) or not torch.isfinite(x).all():raise ValueError('CSMR input')
    x=x.float();gains=gains.float().reshape(-1)
    if len(gains)!=len(x) or not ((gains>=0)&(gains<=1)).all():raise ValueError('CSMR gain')
    spectrum=torch.fft.fftshift(torch.fft.fft2(x,norm='ortho'),dim=(-2,-1))
    fields=1-(1-gains[:,None,None,None])*masks[None]
    if CSMR_PRESERVE_DC:fields[:,:,40,40]=1
    raw=torch.fft.ifft2(torch.fft.ifftshift(spectrum[:,None]*fields[:,:,None],dim=(-2,-1)),norm='ortho').real
    clips=((raw<0)|(raw>1)).float().mean(dim=(2,3,4));views=raw.clamp(0,1)
    if not torch.isfinite(views).all():raise RuntimeError('Nonfinite CSMR views')
    return views,clips
def aligned_margin(logits,binary):
    d=logits[:,0].float()-logits[:,1].float()
    return torch.where(binary==0,d,-d)
def select_worst_view(model,base,binary,masks,gain_range):
    was_training=model.training;model.eval();chosen=[];bands=[];gains_out=[];clips_out=[]
    try:
        with torch.no_grad():
            for start in range(0,len(base),SELECTION_CHUNK_SIZE):
                x=base[start:start+SELECTION_CHUNK_SIZE];y=binary[start:start+SELECTION_CHUNK_SIZE]
                gains=torch.empty(len(x),device=x.device).uniform_(*gain_range)
                views,clips=spectral_views(x,gains,masks);margins=[]
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=use_amp):logits=model(views[:,band])
                    margins.append(aligned_margin(logits,y))
                ix=torch.stack(margins,dim=1).argmin(dim=1);row=torch.arange(len(x),device=x.device)
                chosen.append(views[row,ix].detach());bands.append(ix.detach());gains_out.append(gains.detach());clips_out.append(clips[row,ix].detach())
    finally:model.train(was_training)
    return tuple(torch.cat(z) for z in (chosen,bands,gains_out,clips_out))
def auxiliary_loss(outputs,binary,spoof,lighting,attrs):
    pad,spoof_logits,light_logits,attr_logits=outputs
    ls=F.cross_entropy(spoof_logits,spoof);ll=F.cross_entropy(light_logits,lighting);mask=binary==0
    la=F.binary_cross_entropy_with_logits(attr_logits[mask],attrs[mask]) if mask.any() else attr_logits.sum()*0
    return ls,ll,la
def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all()}
def restore_rng(r):
    random.setstate(r['python']);np.random.set_state(r['numpy']);torch.set_rng_state(r['torch']);torch.cuda.set_rng_state_all(r['cuda'])

def validate_mechanism_spec(name,spec):
    if name not in RUNS or spec!=RUNS[name]:raise RuntimeError('Unknown or changed mechanism run')
    if spec['warmup_epochs'] not in (1,3) or len(spec['gain'])!=2 or not 0<spec['gain'][0]<=spec['gain'][1]<=1:
        raise RuntimeError('Invalid spectral schedule/gain')
    if abs(spec['clean_ce_weight']+spec['worst_ce_weight']-1)>1e-12:raise RuntimeError('CE weights must sum to one')
    if spec['regularizer']=='harmful_margin' and (spec['margin_lambda']!=.1 or spec['consistency_lambda']!=0):raise RuntimeError('Margin run mismatch')
    if spec['regularizer']=='kl_consistency' and (spec['margin_lambda']!=0 or spec['consistency_lambda']!=.1):raise RuntimeError('KL run mismatch')
    if spec['regularizer']=='none' and (spec['margin_lambda']!=0 or spec['consistency_lambda']!=0):raise RuntimeError('CE-only run mismatch')
for _run_id,_spec in RUNS.items():validate_mechanism_spec(_run_id,_spec)
if {k:v for k,v in RUNS['P2_csmr_lambda01'].items() if k!='gain'}!={k:v for k,v in RUNS['P4_mild_gain_lambda01'].items() if k!='gain'}:
    raise RuntimeError('P4 must differ from P2 only in gain')
if {k:v for k,v in RUNS['P2_csmr_lambda01'].items() if k!='warmup_epochs'}!={k:v for k,v in RUNS['P5_warmup3_lambda01'].items() if k!='warmup_epochs'}:
    raise RuntimeError('P5 must differ from P2 only in warm-up')
if ({k:v for k,v in RUNS['P2_csmr_lambda01'].items() if k not in ('regularizer','margin_lambda','consistency_lambda')} !=
    {k:v for k,v in RUNS['P6_generic_kl_lambda01'].items() if k not in ('regularizer','margin_lambda','consistency_lambda')}):
    raise RuntimeError('P6 must differ from P2 only in regularizer')

def mechanism_terms(clean_logits,worst_logits,binary,aux_losses,spec):
    pad_clean=F.cross_entropy(clean_logits,binary)
    pad_worst=F.cross_entropy(worst_logits,binary)
    m_clean=aligned_margin(clean_logits,binary)
    m_worst=aligned_margin(worst_logits,binary)
    target=m_clean.detach();harmful=m_worst<target
    drop=target-m_worst
    if spec['regularizer']=='harmful_margin':
        margin_loss=(F.smooth_l1_loss(m_worst,target,reduction='none')*harmful.float()).mean()
    else:
        margin_loss=pad_clean*0
    if spec['regularizer']=='kl_consistency':
        consistency_loss=F.kl_div(F.log_softmax(worst_logits.float(),dim=1),
                                  F.softmax(clean_logits.float(),dim=1).detach(),reduction='batchmean')
    else:
        consistency_loss=pad_clean*0
    ls,ll,la=aux_losses
    aux_total=.1*ls+.1*ll+la
    classification=spec['clean_ce_weight']*pad_clean+spec['worst_ce_weight']*pad_worst
    total=classification+spec['margin_lambda']*margin_loss+spec['consistency_lambda']*consistency_loss+aux_total
    clean_correct=clean_logits.argmax(dim=1)==binary
    worst_wrong=worst_logits.argmax(dim=1)!=binary
    stats={'harmful_count':int(harmful.sum().detach()),
           'harmful_drop_sum':float(drop.detach()[harmful].sum()),
           'spectral_flip_count':int((clean_correct&worst_wrong).sum().detach()),
           'margin_drop_sum':float(drop.detach().sum())}
    terms={'train_total_loss':total,'pad_clean_loss':pad_clean,'pad_worst_loss':pad_worst,
           'classification_loss':classification,'margin_loss':margin_loss,'consistency_loss':consistency_loss,
           'spoof_loss':ls,'lighting_loss':ll,'attr_loss':la,
           'harmful_fraction':harmful.float().mean(),'clean_margin_mean':m_clean.mean(),
           'worst_margin_mean':m_worst.mean(),'margin_drop_mean':drop.mean()}
    return total,terms,stats

def run_config(name,spec):
    return {'schema':'csmr_mechanism_pilot_10k_v1','run_id':name,'spec':spec,'train_seed':TRAIN_SEED,'data_seed':DATA_SEED,
      'protocol_sha256':PROTOCOL_SHA,'contract_sha256':EXPECTED_CONTRACT_SHA,'resource_provenance_sha256':sha(root/'provenance.json'),
      'train_manifest_sha256':fps['mini_train10k']['file_sha256'],'val_manifest_sha256':fps['mini_val3k']['file_sha256'],
      'test_manifest_sha256':fps['crossdomain_celeba_source_dev_full']['file_sha256'],
      'lcc_manifest_sha256':fps['lcc_official_evaluation_full']['file_sha256'],
      'celeba_bbox_sha256':prov['celeba_bbox_sha256'],'lcc_bbox_sha256':prov['lcc_bbox_sha256'],
      'model_source_sha256':sha(source),'pretrained_checkpoint_sha256':sha(pretrained),
      'initialization':'PAD-pretrained MiniFASNetV2 features; deterministic binary and auxiliary heads',
      'initial_state_sha256':FROZEN_INITIAL_STATE_SHA,
      'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'earliest_early_stop_epoch':EARLIEST_EARLY_STOP_EPOCH,
      'patience':EARLY_STOP_PATIENCE,'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4},
      'scheduler':{'type':'MultiStepLR','milestones':[6,14],'gamma':.2,'step':'after each completed epoch'},
      'label_smoothing':0,'grad_clip':None,'mixup':None,'cutmix':None,'weighted_sampler':None,
      'amp':bool(use_amp),'augmentation':'literature adapted RGB-only augmentation then BGR MiniFAS input',
      'albumentations_version':A.__version__,'augmentation_api':AUGMENTATION_API,
      'bbox_jitter':'20pct scale .95-1.05 shift +/- .05','input':'80x80 BGR float [0,1] 2.7x',
      'spectral':{'centers':CSMR_BAND_CENTERS,'sigma':CSMR_BAND_SIGMA,'preserve_dc':CSMR_PRESERVE_DC,
                  'chunk_size':SELECTION_CHUNK_SIZE,'gain':spec['gain'],'selection':'smallest binary aligned margin; eval no_grad'}}

def prediction_frame(frame,keys,y,z,threshold,kind):
    scores=pad_score(z)
    scored=pd.DataFrame({'key':keys,'three_class_label':y,'pad_logit_score':scores,'pred_real':scores>=threshold,
                         'logit_0':z[:,0],'logit_1':z[:,1]})
    if kind=='val':return scored
    base=frame.copy().rename(columns={'sample_key':'key','path':'key'})
    full=base.merge(scored.drop(columns=['three_class_label']),on='key',how='left',validate='one_to_one',sort=False)
    valid=base.usable_for_pad.astype(bool)
    if full.loc[valid,'pad_logit_score'].isna().any() or full.loc[~valid,'pad_logit_score'].notna().any():
        raise RuntimeError('Candidate/scored prediction accounting mismatch')
    return full

def run_one(name,spec):
    validate_mechanism_spec(name,spec)
    folder=out/'runs'/name;folder.mkdir(parents=True,exist_ok=True)
    cfg=run_config(name,spec);cfg_hash=hash_json(cfg)
    cfg_path=folder/'config.json';last_path=folder/'last_state.pth';best_path=folder/'best.pth';complete_path=folder/'completion.json'
    if cfg_path.exists():
        if read_json(cfg_path)!=cfg:raise RuntimeError(name+' config drift; use a new output directory')
    else:write_json(cfg_path,cfg)
    if hasattr(COPY_AUG,'set_random_seed'):COPY_AUG.set_random_seed(TRAIN_SEED)
    model,audit=init_model({'init':'pretrained','copied':True})
    if audit['initial_state_sha256']!=FROZEN_INITIAL_STATE_SHA:raise RuntimeError('Paired initialization drift')
    audit_path=folder/'initialization_audit.json'
    if audit_path.exists() and read_json(audit_path)!=audit:raise RuntimeError(name+' initialization audit drift')
    write_json(audit_path,audit);model=model.to(device)
    if complete_path.exists():
        done=read_json(complete_path)
        required=[folder/(kind+'_'+suffix) for kind in ('val','test','lcc') for suffix in ('metrics.json','predictions.csv')]
        if done['config_hash']!=cfg_hash or not best_path.is_file() or done['best_checkpoint_sha256']!=sha(best_path) or not all(p.is_file() for p in required):
            raise RuntimeError(name+' completion provenance/output mismatch')
        print(name,'already completed; held-out inference skipped')
        return done
    if best_path.exists() and not last_path.exists():raise RuntimeError('Orphan best checkpoint without resume state')
    optimizer=torch.optim.SGD(model.parameters(),lr=.005,momentum=.9,weight_decay=5e-4)
    scheduler=torch.optim.lr_scheduler.MultiStepLR(optimizer,milestones=[6,14],gamma=.2)
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    history=[];diagnostics=[];best_acer=float('inf');best_auc=-float('inf');best_epoch=0
    no_improve=0;start=1;warmup_metadata=[]
    if last_path.exists():
        if not RESUME:raise RuntimeError(name+' has an incomplete state; enable RESUME or use a clean output root')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if (state.get('schema')!='csmr_mechanism_last_state_v1' or state.get('run_id')!=name or state.get('config_hash')!=cfg_hash or
            state.get('initial_state_sha256')!=audit['initial_state_sha256'] or state.get('resource_provenance_sha256')!=cfg['resource_provenance_sha256']):
            raise RuntimeError('Resume run/config/init/resource mismatch')
        if state['best_epoch']:
            if not best_path.is_file() or sha(best_path)!=state['best_checkpoint_sha256']:raise RuntimeError('Resume best checkpoint mismatch')
            best_saved=torch.load(best_path,map_location='cpu',weights_only=False)
            if state_sha(best_saved['state_dict'])!=state_sha(state['best_state']):raise RuntimeError('Resume best state mismatch')
        if state['warmup_metadata']:
            warmup_path=folder/'warmup.pth'
            if not warmup_path.is_file() or sha(warmup_path)!=state['warmup_metadata'][-1]['checkpoint_sha256']:
                raise RuntimeError('Resume warm-up checkpoint mismatch')
        if state['epoch']<=spec['warmup_epochs']:
            if state['best_epoch']!=0 or state['no_improve_count']!=0:raise RuntimeError('Warm-up contaminated best/patience')
        if state['phase']!=('warmup' if state['epoch']<=spec['warmup_epochs'] else 'spectral'):
            raise RuntimeError('Resume phase mismatch')
        model.load_state_dict(state['model_state']);optimizer.load_state_dict(state['optimizer_state'])
        scheduler.load_state_dict(state['scheduler_state']);scaler.load_state_dict(state['scaler_state'])
        history=state['history'];diagnostics=state['spectral_diagnostics'];best_acer=state['best_acer'];best_auc=state['best_auc']
        best_epoch=state['best_epoch'];no_improve=state['no_improve_count'];warmup_metadata=state['warmup_metadata'];start=state['epoch']+1
        if len(history)!=state['epoch'] or len(diagnostics)!=max(0,state['epoch']-spec['warmup_epochs']):raise RuntimeError('Resume history length mismatch')
        if len(warmup_metadata)!=min(state['epoch'],spec['warmup_epochs']) or no_improve<0:
            raise RuntimeError('Resume warm-up/patience history mismatch')
        if state['epoch']>spec['warmup_epochs'] and best_epoch<=spec['warmup_epochs']:
            raise RuntimeError('Active run lacks an eligible best checkpoint')
        restore_rng(state['rng']);print(name,'resume from epoch',start)
    train_ds=CelebaData(train,True,True);val_dl=loader(CelebaData(val),BATCH_SIZE);masks=radial_masks()
    if start<=MAX_EPOCHS and not (history and history[-1].get('stop_reached')):
        for epoch in range(start,MAX_EPOCHS+1):
            phase='warmup' if epoch<=spec['warmup_epochs'] else 'spectral';active=phase=='spectral'
            model.train();totals=Counter();nseen=0;bands_count=Counter()
            gain_sum=0.0;harmful_count=0;harmful_drop_sum=0.0;flip_count=0;margin_drop_sum=0.0
            for x,_y3,binary,spoof,lighting,attrs,_keys in loader(train_ds,BATCH_SIZE,True,epoch):
                x=x.to(device,non_blocking=True);binary=binary.to(device);spoof=spoof.to(device)
                lighting=lighting.to(device);attrs=attrs.to(device);optimizer.zero_grad(set_to_none=True)
                if active:
                    selected,band,gain,_clip=select_worst_view(model,x,binary,masks,spec['gain'])
                    bands_count.update(band.detach().cpu().tolist());gain_sum+=float(gain.detach().sum())
                with torch.cuda.amp.autocast(enabled=use_amp):
                    clean=model(x,aux=True);pad_clean=F.cross_entropy(clean[0],binary)
                    ls,ll,la=auxiliary_loss(clean,binary,spoof,lighting,attrs)
                    if active:
                        worst=model(selected)
                        loss,terms,stats=mechanism_terms(clean[0],worst,binary,(ls,ll,la),spec)
                    else:
                        zero=pad_clean*0;loss=pad_clean+.1*ls+.1*ll+la
                        terms={'train_total_loss':loss,'pad_clean_loss':pad_clean,'pad_worst_loss':zero,
                               'classification_loss':pad_clean,'margin_loss':zero,'consistency_loss':zero,
                               'spoof_loss':ls,'lighting_loss':ll,'attr_loss':la,'harmful_fraction':zero,
                               'clean_margin_mean':zero,'worst_margin_mean':zero,'margin_drop_mean':zero}
                        stats={'harmful_count':0,'harmful_drop_sum':0.0,'spectral_flip_count':0,'margin_drop_sum':0.0}
                scaler.scale(loss).backward()
                scaler.step(optimizer);scaler.update() # Batch-256 OOM is fatal; no per-run reduction.
                count=len(x);nseen+=count
                for key,value in terms.items():totals[key]+=float(value.detach())*count
                harmful_count+=stats['harmful_count'];harmful_drop_sum+=stats['harmful_drop_sum']
                flip_count+=stats['spectral_flip_count'];margin_drop_sum+=stats['margin_drop_sum']
            val_keys,val_y,val_logits=infer(model,val_dl)
            if val_keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Val ordered-key mismatch')
            val_scores=pad_score(val_logits);threshold=threshold_from_val(val_y,val_scores)
            vm=metrics(val_y,val_scores,threshold,len(val))
            improved=active and (vm['acer']<best_acer-METRIC_TIE_EPS or
                (abs(vm['acer']-best_acer)<=METRIC_TIE_EPS and vm['auc']>best_auc+METRIC_TIE_EPS))
            if active:
                if improved:
                    best_acer=vm['acer'];best_auc=vm['auc'];best_epoch=epoch;no_improve=0
                    save_torch(best_path,{'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                        'epoch':epoch,'run_id':name,'config_hash':cfg_hash,'val_acer':best_acer,'val_auc':best_auc})
                else:no_improve+=1
            else:
                warmup_path=folder/'warmup.pth'
                save_torch(warmup_path,{'state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                    'epoch':epoch,'run_id':name,'config_hash':cfg_hash})
                warmup_metadata.append({'epoch':epoch,'val_metrics':vm,'checkpoint_sha256':sha(warmup_path)})
                write_json(folder/'warmup_metrics.json',{'epochs':warmup_metadata,'last_epoch':epoch})
            lr=optimizer.param_groups[0]['lr']
            row={'epoch':epoch,'phase':phase,'lr':lr,'eligible_for_final_checkpoint':active,
                 'val_acer':vm['acer'],'val_auc':vm['auc'],'val_eer':vm['eer'],'threshold':threshold,
                 'best_so_far':bool(improved),'no_improve_count':no_improve,
                 'clean_ce_weight':spec['clean_ce_weight'] if active else 1.0,
                 'worst_ce_weight':spec['worst_ce_weight'] if active else 0.0,
                 'margin_lambda':spec['margin_lambda'] if active else 0.0,
                 'consistency_lambda':spec['consistency_lambda'] if active else 0.0,
                 'gain_min':spec['gain'][0] if active else None,'gain_max':spec['gain'][1] if active else None,
                 'sampled_gain_mean':gain_sum/nseen if active else None,
                 'low_selected_count':bands_count[0] if active else 0,
                 'mid_selected_count':bands_count[1] if active else 0,
                 'high_selected_count':bands_count[2] if active else 0,
                 'low_selected_fraction':bands_count[0]/nseen if active else 0.0,
                 'mid_selected_fraction':bands_count[1]/nseen if active else 0.0,
                 'high_selected_fraction':bands_count[2]/nseen if active else 0.0,
                 'mean_harmful_margin_drop':harmful_drop_sum/harmful_count if harmful_count else 0.0,
                 'spectral_flip_rate':flip_count/nseen if active else 0.0,
                 **{k:v/nseen for k,v in totals.items()}}
            if active:
                if sum(bands_count.values())!=nseen:raise RuntimeError('Worst-band selection did not cover all samples')
                if abs(row['margin_drop_mean']-margin_drop_sum/nseen)>1e-5:raise RuntimeError('Margin-drop diagnostic mismatch')
                diagnostics.append(dict(row))
            stop=epoch==MAX_EPOCHS or (active and epoch>=EARLIEST_EARLY_STOP_EPOCH and no_improve>=EARLY_STOP_PATIENCE)
            row['stop_reached']=stop;history.append(row)
            scheduler.step() # After completed epoch: milestones 6/14 affect epochs 7/15.
            state={'schema':'csmr_mechanism_last_state_v1','run_id':name,'config_hash':cfg_hash,
                   'resource_provenance_sha256':cfg['resource_provenance_sha256'],
                   'initial_state_sha256':audit['initial_state_sha256'],'epoch':epoch,'phase':phase,
                   'model_state':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                   'best_state':torch.load(best_path,map_location='cpu',weights_only=False)['state_dict'] if best_epoch else None,
                   'best_checkpoint_sha256':sha(best_path) if best_epoch else None,
                   'optimizer_state':optimizer.state_dict(),'scheduler_state':scheduler.state_dict(),
                   'scaler_state':scaler.state_dict(),'rng':rng_state(),'history':history,
                   'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                   'no_improve_count':no_improve,'warmup_metadata':warmup_metadata,
                   'spectral_diagnostics':diagnostics}
            save_torch(last_path,state)
            pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
            pd.DataFrame(diagnostics).to_csv(folder/'spectral_diagnostics.csv',index=False)
            print(name,'epoch',epoch,phase,'Val ACER',vm['acer'],'best',best_epoch,'patience',no_improve,flush=True)
            if stop:break
    if not best_path.is_file() or best_epoch<=spec['warmup_epochs']:raise RuntimeError('No eligible spectral checkpoint')
    best=torch.load(best_path,map_location='cpu',weights_only=False)
    if best['run_id']!=name or best['config_hash']!=cfg_hash or best['epoch']!=best_epoch:
        raise RuntimeError('Best checkpoint policy mismatch')
    model.load_state_dict(best['state_dict']);model.to(device)
    val_keys,val_y,val_logits=infer(model,val_dl)
    if val_keys!=val.sample_key.astype(str).tolist():raise RuntimeError('Frozen Val ordered-key mismatch')
    val_scores=pad_score(val_logits);locked=threshold_from_val(val_y,val_scores)
    val_metrics=metrics(val_y,val_scores,locked,len(val))
    if abs(val_metrics['acer']-best['val_acer'])>1e-6 or abs(val_metrics['auc']-best['val_auc'])>1e-6:
        raise RuntimeError('Frozen best checkpoint does not reproduce selected Val metrics')
    write_json(folder/'val_metrics.json',val_metrics)
    prediction_frame(val,val_keys,val_y,val_logits,locked,'val').to_csv(folder/'val_predictions.csv',index=False)
    freeze={'run_id':name,'config_hash':cfg_hash,'best_checkpoint_sha256':sha(best_path),
            'selected_epoch':best_epoch,'locked_threshold':locked,'val_metrics':val_metrics}
    freeze_path=folder/'evaluation_freeze.json'
    if freeze_path.exists() and read_json(freeze_path)!=freeze:raise RuntimeError('Held-out evaluation freeze mismatch')
    write_json(freeze_path,freeze)
    # Held-out inference happens only after the checkpoint and Val threshold are frozen.
    for kind,scored_frame,ds,keycol in [
      ('test',test.loc[test.usable_for_pad.astype(bool)],CelebaData(test.loc[test.usable_for_pad.astype(bool)]),'sample_key'),
      ('lcc',lcc.loc[lcc.usable_for_pad.astype(bool)],LCCData(lcc),'path')]:
        metrics_path=folder/(kind+'_metrics.json');pred_path=folder/(kind+'_predictions.csv')
        if metrics_path.exists() or pred_path.exists():
            if not metrics_path.is_file() or not pred_path.is_file():raise RuntimeError('Partial held-out output: '+kind)
            print(name,kind,'already evaluated under matching freeze; no repeat inference')
            continue
        keys,y,z=infer(model,loader(ds,BATCH_SIZE),kind=='lcc')
        if keys!=scored_frame[keycol].astype(str).tolist():raise RuntimeError(kind+' scored-key order mismatch')
        candidates=test if kind=='test' else lcc
        mm=metrics(y,pad_score(z),locked,len(candidates))
        write_json(metrics_path,mm)
        prediction_frame(candidates,keys,y,z,locked,kind).to_csv(pred_path,index=False)
    result={'run_id':name,'config_hash':cfg_hash,'selected_epoch':best_epoch,
            'best_checkpoint_sha256':sha(best_path),'locked_threshold':locked,
            'second_lr_decay_reached':bool(history and history[-1]['epoch']>=15),
            'val_auc':val_metrics['auc'],'val_eer':val_metrics['eer'],'val_acer':val_metrics['acer'],
            **{f'{kind}_{metric}':read_json(folder/(kind+'_metrics.json'))[metric]
               for kind in ('test','lcc') for metric in ('auc','eer','acer','detector_coverage')}}
    write_json(complete_path,result)
    return result

def verified_baseline_c(folder,expected_initial_sha):
    folder=Path(folder)
    required=[folder/'config.json',folder/'initialization_audit.json',folder/'completion.json',
              folder/'evaluation_freeze.json',folder/'best.pth',
              *(folder/(kind+'_metrics.json') for kind in ('val','test','lcc'))]
    for path in required:
        if not path.is_file():raise FileNotFoundError('Baseline C reference incomplete: '+str(path))
    cfg=read_json(folder/'config.json');audit=read_json(folder/'initialization_audit.json')
    done=read_json(folder/'completion.json');freeze=read_json(folder/'evaluation_freeze.json')
    expected={'run_id':'C_copy_pretrained','spec':{'init':'pretrained','copied':True,'csmr':False},
      'train_seed':TRAIN_SEED,'data_seed':DATA_SEED,'contract_sha256':EXPECTED_CONTRACT_SHA,
      'train_manifest_sha256':fps['mini_train10k']['file_sha256'],
      'val_manifest_sha256':fps['mini_val3k']['file_sha256'],
      'test_manifest_sha256':fps['crossdomain_celeba_source_dev_full']['file_sha256'],
      'lcc_manifest_sha256':fps['lcc_official_evaluation_full']['file_sha256'],
      'celeba_bbox_sha256':prov['celeba_bbox_sha256'],'lcc_bbox_sha256':prov['lcc_bbox_sha256'],
      'model_source_sha256':sha(source),'pretrained_checkpoint_sha256':sha(pretrained),
      'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,
      'earliest_early_stop_epoch':EARLIEST_EARLY_STOP_EPOCH,'patience':EARLY_STOP_PATIENCE,
      'optimizer':{'name':'SGD','lr':.005,'momentum':.9,'weight_decay':5e-4},
      'scheduler':{'type':'MultiStepLR','milestones':[6,14],'gamma':.2,'step':'after each completed epoch'},
      'label_smoothing':0,'grad_clip':None,
      'augmentation':'literature adapted RGB-only augmentation then BGR MiniFAS input',
      'albumentations_version':A.__version__,'augmentation_api':AUGMENTATION_API,
      'bbox_jitter':'20pct scale .95-1.05 shift +/- .05','input':'80x80 BGR float [0,1] 2.7x','csmr':None}
    if cfg!=expected:raise RuntimeError('Baseline C config differs from frozen copied baseline; omit BASELINE_C_DIR or mount the exact C run')
    if audit.get('initial_state_sha256')!=expected_initial_sha or audit.get('checkpoint_sha256')!=sha(pretrained):
        raise RuntimeError('Baseline C initialization fingerprint mismatch')
    if done.get('run_id')!='C_copy_pretrained' or done.get('config_hash')!=hash_json(cfg) or done.get('best_checkpoint_sha256')!=sha(folder/'best.pth'):
        raise RuntimeError('Baseline C completion/checkpoint mismatch')
    if freeze.get('config_hash')!=hash_json(cfg) or freeze.get('best_checkpoint_sha256')!=sha(folder/'best.pth') or float(freeze.get('locked_threshold'))!=float(done['locked_threshold']):
        raise RuntimeError('Baseline C Val evaluation freeze mismatch')
    row={'run_id':'C_copy_pretrained_reference','reference_only':True,'mechanism':'none','source_directory':str(folder)}
    for kind,expected_n in (('val',3000),('test',67170),('lcc',7580)):
        m=read_json(folder/(kind+'_metrics.json'))
        if m.get('candidate_n')!=expected_n or float(m.get('locked_source_threshold'))!=float(done['locked_threshold']):
            raise RuntimeError('Baseline C metric population/threshold mismatch: '+kind)
        for key,value in m.items():row[kind+'_'+key]=value
    return row,{'directory':str(folder),'config_sha256':hash_json(cfg),'initial_state_sha256':expected_initial_sha,
                'best_checkpoint_sha256':sha(folder/'best.pth')}

reference_model,reference_audit=init_model({'init':'pretrained','copied':True})
FROZEN_INITIAL_STATE_SHA=reference_audit['initial_state_sha256']
del reference_model
experiment={'schema':'csmr_mechanism_pilot_10k_v1','runs':RUNS,'run_filter':RUN_FILTER,
            'data_seed':DATA_SEED,'train_seed':TRAIN_SEED,'protocol_sha256':PROTOCOL_SHA,
            'v2_contract_sha256':EXPECTED_CONTRACT_SHA,'resource_provenance_sha256':sha(root/'provenance.json'),
            'manifest_fingerprints':fps,'model_source_sha256':sha(source),
            'pretrained_checkpoint_sha256':sha(pretrained),'initial_state_sha256':FROZEN_INITIAL_STATE_SHA,
            'augmentation_api':AUGMENTATION_API,
            'albumentations_version':A.__version__}
config_path=out/'experiment_config.json'
if config_path.is_file():
    old=read_json(config_path)
    if {k:v for k,v in old.items() if k!='run_filter'}!={k:v for k,v in experiment.items() if k!='run_filter'}:
        raise RuntimeError('Experiment configuration drift')
write_json(config_path,experiment)
(out/'CSMR_MECHANISM_PILOT_PROTOCOL_v1.md').write_text(PROTOCOL_TEXT)
if sha(out/'CSMR_MECHANISM_PILOT_PROTOCOL_v1.md')!=PROTOCOL_SHA:raise RuntimeError('Exported protocol SHA mismatch')
baseline_row=None;baseline_audit=None
if BASELINE_C_DIR is not None:
    baseline_row,baseline_audit=verified_baseline_c(BASELINE_C_DIR,FROZEN_INITIAL_STATE_SHA)
write_json(out/'provenance.json',{'resource':prov,'experiment_config_sha256':hash_json({k:v for k,v in experiment.items() if k!='run_filter'}),
    'protocol_sha256':PROTOCOL_SHA,'torch':torch.__version__,'cuda':torch.version.cuda,
    'gpu':torch.cuda.get_device_name(0),'albumentations':A.__version__,
    'augmentation_api':AUGMENTATION_API,'opencv':cv2.__version__,'baseline_reference':baseline_audit})
for run_id,spec in RUNS.items():
    if RUN_FILTER is None or run_id in RUN_FILTER:
        try:run_one(run_id,spec)
        except torch.cuda.OutOfMemoryError as exc:
            raise RuntimeError('Batch 256 OOM. Stop; no per-run batch reduction. A changed batch needs a new protocol.') from exc

summary=[]
if baseline_row is not None:summary.append(baseline_row)
for run_id,spec in RUNS.items():
    folder=out/'runs'/run_id
    if not (folder/'completion.json').is_file():continue
    if read_json(folder/'config.json')!=run_config(run_id,spec):raise RuntimeError('Summary config drift: '+run_id)
    completion=read_json(folder/'completion.json')
    if completion['best_checkpoint_sha256']!=sha(folder/'best.pth'):raise RuntimeError('Summary checkpoint drift: '+run_id)
    row={'run_id':run_id,'reference_only':False,'mechanism':spec['regularizer'],
         'gain_min':spec['gain'][0],'gain_max':spec['gain'][1],
         'warmup_epochs':spec['warmup_epochs'],'clean_ce_weight':spec['clean_ce_weight'],
         'worst_ce_weight':spec['worst_ce_weight'],'margin_lambda':spec['margin_lambda'],
         'consistency_lambda':spec['consistency_lambda'],'selected_epoch':completion['selected_epoch'],
         'second_lr_decay_reached':completion['second_lr_decay_reached']}
    for kind in ('val','test','lcc'):
        m=read_json(folder/(kind+'_metrics.json'))
        for key,value in m.items():row[kind+'_'+key]=value
    summary.append(row)
write_json(out/'comparison_summary.json',summary)
pd.DataFrame(summary).to_csv(out/'comparison_summary.csv',index=False)
lookup={row['run_id']:row for row in summary}
contrasts=[('C_copy_pretrained_reference','P1_worst_ce_lambda0','C vs P1: worst-band classification alone'),
 ('P1_worst_ce_lambda0','P2_csmr_lambda01','P1 vs P2: harmful-margin lambda 0.1'),
 ('P1_worst_ce_lambda0','P3_mild_worst_ce_lambda0','P1 vs P3: worst-view CE pressure'),
 ('P2_csmr_lambda01','P4_mild_gain_lambda01','P2 vs P4: spectral gain'),
 ('P2_csmr_lambda01','P5_warmup3_lambda01','P2 vs P5: clean warm-up duration'),
 ('P2_csmr_lambda01','P6_generic_kl_lambda01','P2 vs P6: PAD-margin versus generic consistency')]
report=['# CSMR mechanism pilot 10K','',f'Protocol SHA256: `{PROTOCOL_SHA}`.',
        f'Completed mechanism runs: {sum(not r["reference_only"] for r in summary)}/6.','',
        'All runs use frozen Mini Train10K and Mini Val3K. Official Test and LCC are evaluated only after each Val checkpoint and threshold freeze.',
        'LCC is pilot evidence and is not used to choose a run, threshold, lambda, gain, warm-up, or formulation.','',
        '## Available metrics','',
        '| Run | Val ACER | Test AUC | LCC AUC | LCC EER | LCC coverage |','|---|---:|---:|---:|---:|---:|']
for row in summary:
    report.append(f"| {row['run_id']} | {row['val_acer']:.6f} | {row['test_auc']:.6f} | {row['lcc_auc']:.6f} | {row['lcc_eer']:.6f} | {row['lcc_detector_coverage']:.6f} |")
report+=['','## Planned mechanism contrasts','','Delta means second run minus first run. A positive AUC delta or negative EER/ACER delta describes the measured difference; no run is selected automatically.','']
for left,right,question in contrasts:
    if left in lookup and right in lookup:
        a,b=lookup[left],lookup[right]
        report.append(f"- **{question}**: Δ Val ACER {b['val_acer']-a['val_acer']:+.6f}; Δ Test AUC {b['test_auc']-a['test_auc']:+.6f}; Δ LCC AUC {b['lcc_auc']-a['lcc_auc']:+.6f}; Δ LCC EER {b['lcc_eer']-a['lcc_eer']:+.6f}.")
    else:
        report.append(f'- **{question}**: pending one or both outputs.')
report+=['','Diagnostics are in each run\'s `spectral_diagnostics.csv`; warm-up epochs never enter final checkpoint selection or active patience.','']
(out/'mechanism_report.md').write_text('\n'.join(report))
print('Completed P runs:',sum(not r['reference_only'] for r in summary),'of 6; output:',out)
